## Etape 1 : Audit et nettoyage

In [ ]:
import pandas as pd

df = pd.read_csv("textes_films_horreur_comedie.csv")

In [2]:
print(f"{df.cible.sum()} cibles 1 (horreur), {len(df.cible) - df.cible.sum()} cibles 0 (comédies)")

1200 cibles 1 (horreur), 1200 cibles 0 (comédies)


In [3]:
# suppression des espaces intempestifs
import re
df["titre"] = df["titre"].apply(lambda str: re.sub(r"  +"," ",str.strip()))
df["resume"] = df["resume"].apply(lambda str: re.sub(r"  +"," ",str.strip()))

In [4]:
# gestion des doublons
print(len(df.value_counts()[df.value_counts() > 1]),"films en doublon")
df.drop_duplicates(inplace=True)

51 films en doublon


In [5]:
print(f"{df.cible.sum()} cibles 1 (horreur), {len(df.cible) - df.cible.sum()} cibles 0 (comédies)")

1151 cibles 1 (horreur), 1195 cibles 0 (comédies)


54 lignes en doublon ont donc été supprimées (3 lignes étaient même triplées).

In [6]:
print(len(df.value_counts("titre")[df.value_counts("titre") > 1]),"titres de films en doublon")

56 titres de films en doublon


In [7]:
df[df["titre"]=="Dracula"]

,titre,resume,cible
1126,Dracula,This version of Dracula is closely based on Br...,1
1509,Dracula,After a harrowing ride through the Carpathian ...,1
2086,Dracula,"When a ship is wrecked off Whitby, the only su...",1


On suppose donc qu'il s'agit de films différents, même s'ils ont le même titre.

## Etape 2 : Découpage du jeu de données
Séparer les textes en trois parties : 60 % apprentissage, 20% test, 20% production

In [13]:
from sklearn.model_selection import train_test_split
train, testProd, y_train, y_testProd = train_test_split(df["resume"], df["cible"], test_size=0.4, stratify=df["cible"], random_state=42)
test, prod, y_test, y_prod = train_test_split(testProd, y_testProd, test_size=0.5, stratify=y_testProd, random_state=42)
print("Répartition train / test / prod :")
print(f"* comédies : {len(y_train[y_train==0])} / {len(y_test[y_test==0])} / {len(y_prod[y_prod==0])}")
print(f"* horreurs : {len(y_train[y_train==1])} / {len(y_test[y_test==1])} / {len(y_prod[y_prod==1])}")

Répartition train / test / prod :
* comédies : 717 / 239 / 239
* horreurs : 690 / 230 / 231


## Etape 3 : Vectorisation du texte TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words="english", max_features=15000)
X_train = vectorizer.fit_transform(train.apply(lambda str:str.lower()))
X_test = vectorizer.transform(test.apply(lambda str:str.lower()))
X_prod = vectorizer.transform(prod.apply(lambda str:str.lower()))

In [97]:
print("Taille du vocabulaire :",X_train.shape[1])

Taille du vocabulaire : 15000


## Etape 4 : Comparaison de 3 modèles

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from keras.utils import set_random_seed
from keras import Sequential
from keras.layers import Input, Dense, Dropout, GlobalMaxPooling1D, Conv1D, Embedding
from keras.optimizers import Adam
import numpy as np

# modele Logistic Regression :
modele = LogisticRegression(
    C=2,
    max_iter=2000
).fit(X_train, y_train)
f1_LogReg = 0
meilleur_seuil = 0
y_proba = modele.predict_proba(X_test)[:, 1]
for seuil in np.arange(0.40, 0.61, 0.01):
    y_pred = (y_proba >= seuil).astype(int)
    f1 = f1_score(np.array(y_test).ravel(), y_pred)
    if f1 > f1_LogReg:
        f1_LogReg = f1
        meilleur_seuil = seuil
# modele Bayes naïf :
f1_Bayes = 0 
best_alpha = 0
for alpha in [0.01, 0.1, 0.5, 1, 2, 5]:
    modele = MultinomialNB(alpha=alpha)
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    if f1 > f1_Bayes:
        f1_Bayes = f1
        best_alpha = alpha
# réseau simple :
def reseauEntraine(X_tr, y_tr, neurones=32, couches=1, dropout=0.2, lr=1e-3, batch=32, seed=42):
    set_random_seed(seed)
    m = Sequential([Input(shape=(X_tr.shape[1],))])
    for _ in range(couches):
        m.add(Dense(neurones, activation="relu"))
        if dropout:
            m.add(Dropout(dropout))
    m.add(Dense(1, activation="sigmoid"))
    m.compile(optimizer=Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(X_tr, y_tr, validation_split=0.2, epochs=40, batch_size=batch, verbose=0)
    return m
f1_reseau = 0
nb = 0
for seed in range(5):
    modele = reseauEntraine(X_train, y_train, seed=seed)
    y_pred = modele.predict(X_test)
    f1_reseau += f1_score(y_test, (y_pred.ravel() >= 0.5).astype(int))
    nb += 1
f1_reseau /= nb
# bilan
print(f"Modèle Logistic Regression : {f1_LogReg} meilleur_seuil : {meilleur_seuil}, Modèle Bayes naïf : {f1_Bayes}, best_alpha : {best_alpha}, Réseau : {f1_reseau}")

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
Modèle Logistic Regression : 0.8716814159292036 meilleur_seuil : 0.5000000000000001, Modèle Bayes naïf : 0.8649789029535865, best_alpha : 0.5, Réseau : 0.8504480700933789
